# Weak References in Python — Backend Interview Guide

---

## 1. Simple Intuition First

Imagine you hired a personal assistant and gave them a key to your house.
As long as **you** have that key (strong reference), the house (object) stays in memory.

Now imagine you gave someone **a copy of the address** — not a key.  
If you destroy the house, they can knock but nobody answers. That's a **weak reference**.

In Python:
- **Normal (strong) reference** → object stays alive as long as reference exists
- **Weak reference** → object CAN be garbage collected even if weak references exist

```python
import weakref

class HeavyObject:
    def __init__(self, name):
        self.name = name

obj = HeavyObject("cache_entry")   # strong reference
ref = weakref.ref(obj)             # weak reference — doesn't keep obj alive

print(ref())   # → <HeavyObject ...>  (obj still alive)

del obj        # destroy strong reference

print(ref())   # → None  (GC already collected it)
```

**Key takeaway:** Weak references let you **observe** an object without **owning** it.

---

## 2. Why This Topic Exists

### The Real Problem: Memory Leaks via Unintended Retention

In backend systems, you often build in-memory registries, caches, or observer lists.
If you store strong references to objects in these structures:

- The objects **never get garbage collected** even when they're no longer needed
- Memory keeps growing → **OOM kills in production**

### Classic Leak Example

```python
# A registry holding all active sessions
registry = {}

class UserSession:
    def __init__(self, user_id):
        self.user_id = user_id
        registry[user_id] = self   # strong reference!

# Even if all caller code drops the session object,
# registry holds it alive → MEMORY LEAK
```

### Why Companies Care

| Scenario | Problem Without Weak Refs |
|---|---|
| In-memory cache | Cached objects never freed, RAM grows unboundedly |
| Observer/callback pattern | Listeners hold referenced objects alive after logout |
| Object registry/pool | Stale entries never cleaned up |
| Circular reference between objects | GC can't collect either |

---

## 3. Core Concepts

### 3.1 `weakref.ref` — Basic Weak Reference

```python
import weakref

ref = weakref.ref(obj)   # create weak ref
ref()                    # dereference → returns object or None
```

Always check `ref() is not None` before using.

---

### 3.2 `weakref.WeakValueDictionary` — Most Interview-Relevant

A dict where **values** are weak references.  
When the object is GC'd, the key-value pair is **automatically removed**.

```python
from weakref import WeakValueDictionary

cache = WeakValueDictionary()

class DBConnection:
    def __init__(self, conn_id):
        self.conn_id = conn_id

conn = DBConnection("conn-101")
cache["conn-101"] = conn

print(cache["conn-101"])  # works
del conn
print(dict(cache))        # {} — auto-cleaned
```

**Use case:** Object registries, soft caches.

---

### 3.3 `weakref.WeakKeyDictionary`

Keys are weak references. Used when you want to attach metadata to an object **without** keeping it alive.

```python
# Attach per-object metadata without retaining objects
metadata = WeakKeyDictionary()
metadata[my_object] = {"created_at": "..."}
```

---

### 3.4 `weakref.WeakSet`

A set where members are weakly held. Useful for subscriber/observer lists.

```python
subscribers = weakref.WeakSet()
subscribers.add(some_listener)
# If listener is GC'd, it auto-removes from set
```

---

### 3.5 Finalizers — `weakref.finalize`

Run cleanup code when an object is about to be garbage collected.

```python
def cleanup(name):
    print(f"Cleaning up {name}")

obj = HeavyObject("db_connection")
weakref.finalize(obj, cleanup, obj.name)
del obj   # triggers: "Cleaning up db_connection"
```

---

### 3.6 What Objects Support Weak References?

Not all objects support weak refs. Supported by default:
- User-defined classes (unless `__slots__` without `__weakref__`)
- Functions, methods

**NOT supported:**
- `int`, `str`, `tuple`, `list`, `dict` (built-in types)
- Classes with `__slots__` that don't include `'__weakref__'`

---

## 4. Production-Level Example

### Scenario: In-Memory Cache for Auth Tokens / Active Sessions

```python
import weakref
import time

class AuthSession:
    def __init__(self, user_id: str, token: str):
        self.user_id = user_id
        self.token = token
        self.created_at = time.time()

    def is_valid(self) -> bool:
        return (time.time() - self.created_at) < 3600  # 1 hour TTL

class SessionRegistry:
    """
    Tracks active sessions WITHOUT preventing GC.
    When callers drop the session object (logout, timeout),
    it auto-removes from registry — no manual cleanup needed.
    """
    def __init__(self):
        self._sessions = weakref.WeakValueDictionary()

    def register(self, session: AuthSession):
        self._sessions[session.user_id] = session

    def get(self, user_id: str) -> AuthSession | None:
        return self._sessions.get(user_id)

    def active_count(self) -> int:
        return len(self._sessions)


# Usage
registry = SessionRegistry()

session = AuthSession("user_123", "tok_abc")
registry.register(session)

print(registry.get("user_123"))  # Returns session
print(registry.active_count())   # 1

# User logs out — caller drops the object
del session

# Registry auto-cleaned — no stale entry
print(registry.get("user_123"))  # None
print(registry.active_count())   # 0
```

### Where This Appears in Real Systems

- **Django / Flask** — per-request object registries
- **SQLAlchemy Identity Map** — tracks loaded DB objects without leaking
- **Celery / Task queues** — worker pools track active tasks weakly
- **Event systems** — pub/sub subscriber lists use WeakSet so dead listeners don't accumulate

---

## 5. Internal Working

### CPython Memory Model

CPython uses **reference counting** as the primary GC strategy.

```
object in memory → has a reference count (ob_refcnt)
strong ref added → ob_refcnt++
strong ref deleted → ob_refcnt--
ob_refcnt hits 0 → object freed immediately
```

### What Weak References Do Internally

1. `weakref.ref(obj)` creates a `weakref` object pointing to `obj`
2. CPython maintains a **list of weak references** attached to the object (not counted in `ob_refcnt`)
3. When `ob_refcnt` hits 0:
   - Before freeing: Python **nullifies all weak refs** pointing to this object
   - Then frees the object
4. Calling `ref()` after this → returns `None`

```
Strong refs: [caller_code]    → ob_refcnt = 1
Weak refs:   [registry, logs] → NOT counted

del caller_code
→ ob_refcnt = 0
→ CPython nullifies all weak refs
→ object freed
→ registry.get(key) → None (auto-cleaned in WeakValueDict)
```

### GC Cyclic Collector (for circular references)

For cases where `ob_refcnt` never hits 0 due to cycles, Python runs a **cyclic GC** periodically.
Weak references still work — when cyclic GC collects objects, weak refs are nullified first.

---

## 6. Most Common Interview Questions

---

### Q1: What is a weak reference and when would you use it?

**Strong answer:**
> "A weak reference is a reference to an object that doesn't prevent it from being garbage collected. You use it when you want to track or observe an object without being responsible for keeping it alive. Classic production use case: a cache or object registry where you don't want stale entries to cause memory leaks."

**Follow-up:** "How do you safely dereference a weak ref?"
> Always check: `obj = ref(); if obj is not None: use(obj)`

**Common mistake:** Candidates say "it's for circular references" — that's true but not the primary use case. Lead with caching/registry.

---

### Q2: What's the difference between `weakref.ref` and `WeakValueDictionary`?

**Strong answer:**
> "`weakref.ref` is a raw pointer — you manage dereferencing and cleanup manually. `WeakValueDictionary` is a batteries-included dict that auto-removes entries when values are GC'd. In production you almost always want `WeakValueDictionary` over raw `weakref.ref` because you don't want to litter your code with `None` checks and manual key deletion."

---

### Q3: How does SQLAlchemy use weak references?

**Strong answer:**
> "SQLAlchemy's Session maintains an Identity Map — a dict of `{primary_key: object}` for all loaded ORM objects. It uses `WeakValueDictionary` so if application code drops all references to a loaded model instance, the session doesn't retain it. This prevents the session from accumulating all objects ever loaded, which would be catastrophic for long-running services."

**Follow-up:** "What if you want to ensure an object stays loaded?"
> Keep a strong reference in your own scope or use `session.expunge` to explicitly remove it.

---

### Q4: Can you use weak references with all Python objects?

**Strong answer:**
> "No. Built-in types like `int`, `str`, `list`, `dict`, and `tuple` don't support weak references. User-defined classes support it by default via `__weakref__`. If a class uses `__slots__`, you must explicitly include `'__weakref__'` in the slots list, otherwise `weakref.ref()` raises a `TypeError`."

**Common mistake:** Candidates forget about `__slots__` — this is a common gotcha asked at senior levels.

---

### Q5: Weak references vs just using `None` to signal deletion?

**Strong answer:**
> "Setting to `None` is a manual process — someone has to remember to do it. Weak references let Python's GC do the cleanup automatically, without any coordination. In a large system with many components holding references, you can't reliably orchestrate manual `None` assignments. Weak refs are fire-and-forget."

---

## 7. Senior-Level Understanding

### When NOT to Use Weak References

| Situation | Reason to Avoid |
|---|---|
| You WANT the cache to keep objects alive | Use a proper LRU cache (`functools.lru_cache`) |
| Objects are frequently created/destroyed | Constant GC pressure hurts throughput |
| You need deterministic object lifetimes | Weak refs depend on GC timing, not your logic |
| Using PyPy or non-CPython | GC behavior differs — weak refs may not nullify at the same point |

### Performance Implications

- Dereferencing a weak ref has **slight overhead** vs a direct attribute access — not a bottleneck, but don't use it in hot loops unnecessarily
- `WeakValueDictionary` has **locking overhead** for thread-safety during cleanup callbacks
- Finalizers (`weakref.finalize`) add overhead and can **delay GC** if callbacks are expensive

### Scalability Considerations

- Works well for single-process Python backends
- In multi-process setups (Gunicorn, Celery workers), **each process has its own heap** — weak refs don't span processes
- In distributed systems, weak references are **irrelevant** — use Redis TTL or explicit eviction instead

### Common Production Issues

1. **Premature collection:** Temporary object created inline has no strong ref → GC'd immediately
   ```python
   # BUG: obj created inline, no strong ref, GC'd before use
   registry[key] = weakref.ref(MyObject())   # MyObject collected right away!
   ```

2. **Thread safety:** Multiple threads dereferencing the same weak ref → race condition. Always store `ref()` result in a local variable.

---

## 8. Comparison: Strong vs Weak vs Soft References

| Property | Strong Reference | Weak Reference | Note |
|---|---|---|---|
| Prevents GC | ✅ Yes | ❌ No | Python has no `SoftReference` like Java |
| Auto-cleanup on GC | ❌ Manual | ✅ Automatic | WeakValueDict auto-removes |
| Safe to dereference | Always | Must check `None` | `ref()` returns None if GC'd |
| Supports all types | ✅ Yes | ❌ No (no builtins) | int/str/list not supported |
| Typical use | Everything | Cache, Registry | |

### Weak References vs `functools.lru_cache`

| | `WeakValueDictionary` | `lru_cache` |
|---|---|---|
| Eviction | GC decides | Size-based (LRU) |
| Object lifetime | GC-controlled | Cache-controlled |
| Thread-safe | Mostly | Yes |
| Best for | Object registries | Memoization / computed values |

---

## 9. Production Debugging / Failure Cases

### Failure: Premature GC / "Object disappears"

**Symptom:** `ref()` returns `None` unexpectedly  
**Cause:** No strong reference is holding the object alive  
**Debug:**
```python
import gc
gc.collect()  # force GC and observe what gets collected
# Use objgraph library to visualize reference graph
import objgraph
objgraph.show_refs([my_object], max_depth=3)
```

---

### Failure: Memory Leak Despite Using Weak References

**Symptom:** RAM still grows  
**Cause 1:** Using regular dict but intended `WeakValueDictionary`  
**Cause 2:** A strong reference exists somewhere (closures, globals, thread locals)  
**Debug:**
```python
import tracemalloc
tracemalloc.start()
# ... run workload
snapshot = tracemalloc.take_snapshot()
top_stats = snapshot.statistics('lineno')
for stat in top_stats[:10]:
    print(stat)
```

---

### Failure: Race Condition in Multithreaded Code

```python
# BUG — not thread safe
if ref() is not None:
    ref().do_something()   # ref() might return None here if GC ran between lines

# CORRECT
obj = ref()
if obj is not None:
    obj.do_something()
```

---

### Monitoring Guidance

- Track memory with `tracemalloc` or `memory_profiler` in staging
- Alert on RSS memory growth over time (suggests leak)
- Use `gc.get_objects()` to count live instances of suspect classes
- In production: expose a `/debug/gc` endpoint (internal only) that returns `gc.get_count()` and instance counts

---

## 10. Important Code Examples

### Pattern 1: Safe Weak Ref Access

```python
import weakref

class ConnectionPool:
    def __init__(self):
        self._active: dict[str, weakref.ref] = {}

    def track(self, conn_id: str, conn):
        self._active[conn_id] = weakref.ref(conn)

    def get(self, conn_id: str):
        ref = self._active.get(conn_id)
        if ref is None:
            return None
        obj = ref()           # dereference — store in local variable
        if obj is None:
            del self._active[conn_id]   # clean up stale key
        return obj
```

---

### Pattern 2: Observer / Event Bus Without Memory Leak

```python
import weakref

class EventBus:
    def __init__(self):
        self._listeners = weakref.WeakSet()

    def subscribe(self, listener):
        self._listeners.add(listener)

    def emit(self, event: dict):
        for listener in list(self._listeners):  # snapshot to avoid mutation during iteration
            listener.on_event(event)

class OrderProcessor:
    def on_event(self, event: dict):
        print(f"Processing order event: {event}")

bus = EventBus()
processor = OrderProcessor()
bus.subscribe(processor)
bus.emit({"type": "order_created", "id": "ord_001"})

del processor   # unsubscribes automatically — no manual bus.unsubscribe() needed
bus.emit({"type": "order_created", "id": "ord_002"})  # no listener, no error
```

---

### Pattern 3: Class with `__slots__` + Weak Ref Support

```python
class CacheEntry:
    __slots__ = ('key', 'value', '__weakref__')   # must include __weakref__

    def __init__(self, key, value):
        self.key = key
        self.value = value

entry = CacheEntry("user:123", {"name": "Alice"})
ref = weakref.ref(entry)   # works because __weakref__ is in __slots__
```

---

## 11. Revision Notes — Must-Know for Interviews

```
WHAT       → Reference that doesn't prevent garbage collection
MODULE     → import weakref
KEY TOOLS  → weakref.ref, WeakValueDictionary, WeakSet, finalize

CORE RULE  → Always: obj = ref(); if obj is not None: use(obj)

TOP USE CASES (production)
  → In-memory object registries (auth sessions, connections)
  → Observer / pub-sub systems (WeakSet of listeners)
  → SQLAlchemy Identity Map (WeakValueDictionary)

LIMITS
  → int, str, list, dict NOT supported
  → __slots__ classes need '__weakref__' slot explicitly

COMMON INTERVIEW TRICK
  → "Can you use weakref on a list?" → NO
  → "What happens with __slots__?" → Must add '__weakref__'
  → "Difference from lru_cache?" → GC-controlled vs size-controlled eviction

RED FLAGS IN CODE REVIEW
  → registry[k] = weakref.ref(SomeClass())   ← GC'd immediately, no strong ref
  → if ref() is not None: ref().method()    ← race condition, not thread safe

INTERVIEWER FAVOURITE FOLLOW-UPS
  → "Where does CPython actually use this?" → cyclic GC, reference tracking
  → "How does SQLAlchemy avoid memory leaks?" → WeakValueDictionary identity map
  → "What's the production risk?" → premature collection vs memory leak (tradeoff)
```
